# Melhores configurações do CPAP

Qual pressão, umidade e EPR deram as melhores noites, pelas métricas do fluxo bruto (`night_metrics`, ver `03_night_quality.ipynb`).

- **Índice composto:** média dos z-scores de `surge_index`, `event_index` (`hypopnea30_index` + `rera_like_index`) e `stable_pct` com sinal invertido, calculados sobre todas as noites comparáveis. **Menor é melhor.**
- **Cobertura:** as métricas do fluxo só existem de 7,0 a 9,0 cmH2O. As noites de 5,8 a 6,6 são anteriores ao BRP ou caíram na parte corrompida do cartão; para elas só há o AHI da ResMed.
- **Confusão com o tempo:** cada configuração foi usada num período diferente. Por isso as regressões incluem a tendência no tempo (`days`) e a umidade do quarto (`amb_humidity_50`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from sleep_analysis.db import connect

MIN_SLEEP_HOURS = 4
MAX_LEAK_95_L_MIN = 24
MIN_NIGHTS_PER_SETTING = 5
# +1: maior é pior; -1: maior é melhor
COMPOSITE = {"surge_index": 1, "event_index": 1, "stable_pct": -1}

db = connect()
nights = db.sql("SELECT * FROM nights ORDER BY night").df()
metrics = db.sql("""
    SELECT m.*, n.pressure, n.epr_enabled, n.epr_level, n.humidity_level, n.ahi, n.rin,
           n.leak_95_l_min, n.amb_humidity_50, n.hum_temp_50
    FROM night_metrics m
    JOIN nights n USING (night)
    ORDER BY night
""").df()
metrics["event_index"] = metrics["hypopnea30_index"] + metrics["rera_like_index"]

comparable = metrics[
    (metrics["sleep_hours"] >= MIN_SLEEP_HOURS) & (metrics["leak_95_l_min"] < MAX_LEAK_95_L_MIN)
].copy()
for column, direction in COMPOSITE.items():
    comparable[f"z_{column}"] = direction * (comparable[column] - comparable[column].mean()) / comparable[column].std()
comparable["composite"] = comparable[[f"z_{column}" for column in COMPOSITE]].mean(axis=1)
comparable["days"] = (comparable["night"] - comparable["night"].min()).dt.days
comparable["epr"] = comparable["epr_enabled"].astype(int)

print(
    f"{len(comparable)} noites comparáveis com fluxo de 25 Hz, "
    f"de {comparable['night'].min():%d/%m/%Y} a {comparable['night'].max():%d/%m/%Y}"
)
coverage = (
    nights.groupby("pressure").size().to_frame("noites")
    .join(comparable.groupby("pressure").size().rename("com fluxo"))
    .fillna(0).astype(int)
)
coverage.T

## Ranking das configurações

Mediana de cada métrica por combinação de pressão, EPR e umidade, só com combinações de pelo menos 5 noites. Ordenado pelo índice composto.

In [ ]:
SETTING = ["pressure", "epr_enabled", "humidity_level"]
SUMMARY_COLUMNS = ["composite", "surge_index", "hypopnea30_index", "rera_like_index", "flow_limited_pct", "stable_pct", "ahi"]

grouped = comparable.groupby(SETTING)
by_setting = grouped[SUMMARY_COLUMNS].median().round(2).assign(
    n=grouped.size(),
    first=grouped["night"].min().dt.strftime("%d/%m/%y"),
    last=grouped["night"].max().dt.strftime("%d/%m/%y"),
)
by_setting[by_setting["n"] >= MIN_NIGHTS_PER_SETTING].sort_values("composite")

## Umidade

À esquerda, o índice composto por nível de umidade (cores = pressão). À direita, contra a temperatura da câmara do umidificador, que mede quanto ele realmente aqueceu a água naquela noite (depende do nível e do quarto).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
rng = np.random.default_rng(0)
for pressure, group in comparable.groupby("pressure"):
    jitter = rng.uniform(-0.15, 0.15, len(group))
    axes[0].scatter(group["humidity_level"] + jitter, group["composite"], s=14, alpha=0.6, label=f"{pressure:.1f}")
medians = comparable.groupby("humidity_level")["composite"].median()
axes[0].plot(medians.index, medians.values, "k-o", label="mediana")
axes[0].set(xlabel="Nível de umidade", ylabel="Índice composto (menor é melhor)", xticks=medians.index)
axes[0].legend(title="cmH2O", fontsize=8, ncol=2)

axes[1].scatter(comparable["hum_temp_50"], comparable["composite"], s=14, alpha=0.5, color="gray")
bins = pd.cut(comparable["hum_temp_50"], bins=range(15, 60, 5))
binned = comparable.groupby(bins, observed=True)["composite"].agg(["median", "size"])
binned = binned[binned["size"] >= MIN_NIGHTS_PER_SETTING]
axes[1].plot([interval.mid for interval in binned.index], binned["median"], "k-o")
axes[1].set(xlabel="Temperatura da câmara do umidificador, mediana da noite (°C)", ylabel="Índice composto")
fig.tight_layout()

## Regressão: efeito de cada configuração

Regressão linear com erros robustos (HC3), controlando vazamento, umidade do quarto e tendência no tempo. Cada célula mostra o efeito e o intervalo de confiança de 95%; se o intervalo cruza o zero, o efeito não se distingue do acaso.

In [ ]:
TARGETS = {
    "composite": "Índice composto",
    "surge_index": "Surtos/h",
    "event_index": "Hipopneias 30% + RERA-like/h",
    "stable_pct": "Janelas estáveis (%)",
    "flow_limited_pct": "Limitação de fluxo (%)",
}
FORMULA = "pressure + humidity_level + epr + leak_95_l_min + amb_humidity_50 + days"
TERMS = {"pressure": "+1 cmH2O", "humidity_level": "+1 nível de umidade", "epr": "EPR ligado", "days": "+30 dias"}
TERM_SCALE = {"days": 30}

rows = []
for target, label in TARGETS.items():
    fit = smf.ols(f"{target} ~ {FORMULA}", data=comparable).fit(cov_type="HC3")
    ci = fit.conf_int()
    for term, term_label in TERMS.items():
        scale = TERM_SCALE.get(term, 1)
        low, high = ci.loc[term] * scale
        rows.append({
            "métrica": label,
            "variável": term_label,
            "efeito": f"{fit.params[term] * scale:+.2f} [{low:+.2f}, {high:+.2f}]",
        })
pd.DataFrame(rows).pivot(index="variável", columns="métrica", values="efeito")[list(TARGETS.values())].loc[list(TERMS.values())]

## Pressão: existe um ótimo?

Curva quadrática em função da pressão, com as demais variáveis fixas na mediana. Se o termo quadrático for positivo e significativo, o vértice é a pressão ótima estimada, inclusive entre valores não testados. Os pontos são as medianas observadas (tamanho = número de noites).

O AHI e o índice de RERA da ResMed cobrem de 5,8 a 9,0 cmH2O, porque vêm do `STR.edf`.

In [ ]:
device = nights[
    (nights["usage_hours"] >= MIN_SLEEP_HOURS)
    & (nights["epr_enabled"] == 0)
    & (nights["leak_95_l_min"] < MAX_LEAK_95_L_MIN)
]
CURVES = {
    "Índice composto (fluxo)": (comparable, "composite", ["humidity_level", "epr", "leak_95_l_min"]),
    "AHI ResMed (EPR desligado)": (device, "ahi", ["humidity_level", "leak_95_l_min"]),
    "RERA ResMed/h (EPR desligado)": (device, "rin", ["humidity_level", "leak_95_l_min"]),
}

fig, axes = plt.subplots(1, len(CURVES), figsize=(15, 4))
rows = []
for ax, (title, (data, target, controls)) in zip(axes, CURVES.items()):
    fit = smf.ols(f"{target} ~ pressure + I(pressure ** 2) + {' + '.join(controls)}", data=data).fit(cov_type="HC3")
    grid = pd.DataFrame({"pressure": np.round(np.arange(data["pressure"].min(), data["pressure"].max() + 0.01, 0.1), 1)})
    for column in controls:
        grid[column] = data[column].median()
    prediction = fit.get_prediction(grid).summary_frame(alpha=0.05)
    ax.fill_between(grid["pressure"], prediction["mean_ci_lower"], prediction["mean_ci_upper"], alpha=0.2)
    ax.plot(grid["pressure"], prediction["mean"])
    observed = data.groupby("pressure")[target].agg(["median", "size"])
    ax.scatter(observed.index, observed["median"], s=observed["size"] * 3, color="k", alpha=0.6)
    ax.set(title=title, xlabel="Pressão (cmH2O)")

    linear, quadratic = fit.params["pressure"], fit.params["I(pressure ** 2)"]
    rows.append({
        "métrica": title,
        "curva": "tem mínimo" if quadratic > 0 else "tem máximo (sem ótimo)",
        "vértice (cmH2O)": -linear / (2 * quadratic),
        "p do termo quadrático": fit.pvalues["I(pressure ** 2)"],
        "melhor no intervalo testado": grid.loc[prediction["mean"].idxmin(), "pressure"],
    })
fig.tight_layout()
pd.DataFrame(rows).set_index("métrica").round(2)

## Projeção por nível de umidade a 7,0 cmH2O

Previsão da regressão acima para cada nível de umidade, com EPR desligado, sem vazamento, umidade do quarto mediana e na data mais recente. Níveis não testados a 7,0 cmH2O são **extrapolação**: a reta supõe que o ganho continua, o que não está garantido.

In [ ]:
scenario = pd.DataFrame({"humidity_level": [3.0, 4.0, 5.0, 6.0, 7.0]}).assign(
    pressure=7.0,
    epr=0,
    leak_95_l_min=0.0,
    amb_humidity_50=comparable["amb_humidity_50"].median(),
    days=comparable["days"].max(),
)
tested = comparable.loc[comparable["pressure"] == 7.0, "humidity_level"].unique()

projection = pd.DataFrame(index=scenario["humidity_level"].rename("umidade"))
for target, label in TARGETS.items():
    fit = smf.ols(f"{target} ~ {FORMULA}", data=comparable).fit(cov_type="HC3")
    frame = fit.get_prediction(scenario).summary_frame(alpha=0.05)
    projection[label] = [
        f"{mean:.2f} [{low:.2f}, {high:.2f}]"
        for mean, low, high in zip(frame["mean"], frame["mean_ci_lower"], frame["mean_ci_upper"])
    ]
projection["testado a 7,0"] = projection.index.isin(tested)
projection

## Como ler

- Associação, não experimento: as configurações foram trocadas em sequência, não sorteadas. A regressão controla a tendência linear no tempo, mas não mudanças pontuais (estação, rotina, congestão nasal).
- Para confirmar um efeito, alternar a configuração em blocos de 3 a 4 noites (ex.: umidade 5 e 6) com a pressão fixa, e rodar este notebook de novo.
- Antes da cirurgia, manter a configuração atual para não misturar o efeito da cirurgia com o de uma troca de ajuste.